# Classwork: The Analyst's Toolkit

**Names:** _(pair)_

After learning about different tools (or a toolkit-- Descriptive statistics, Correlation, Regression/Multiple regression, Chi-square test) of working with data in the previous Mods, in this acitivity you will apply your real skills on the data you have.  
  
  Suppose your company hands you `customers.csv` (40 customers: `age`, `income`, `spending`, `region`, `subscribed`) and a list of questions. You are the analyst. Answer each one with the right tool/technique you decide to apply.

Work in pairs. Write answers in the markdown cells provided.

## Setup
Import the libraries you'll need and load the data.

In [1]:
# import pandas (and numpy, scipy.stats, statsmodels.api as you need them)
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
# load customers.csv 
df = pd.read_csv('./customers.csv')
# take a first look (head, info)
head = df.head(10)
print(f"--------------------------HEAD--------------------------------\n {head}")
info = df.info()
print(info)


--------------------------HEAD--------------------------------
    customer_id  age  income  spending region subscribed
0            1   25    61.3      39.1   West         No
1            2   25    54.9      42.1  South        Yes
2            3   51   105.1      59.9  South         No
3            4   39     NaN      49.9  South         No
4            5   43    72.4      48.3   East         No
5            6   44    84.7      47.1   East         No
6            7   48    92.9      60.0   East        Yes
7            8   21    53.3      36.0  South         No
8            9   39    71.9      48.9   East        Yes
9           10   25    70.4      42.1   east        Yes
<class 'pandas.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   customer_id  40 non-null     int64  
 1   age          40 non-null     int64  
 2   income       38 non-null     float64
 3   spending     4

## Part 0: Know your data, then clean it

This data is dirty on purpose. Find and fix **three** problems before you analyze anything.

In [2]:
# HINT 3: one value is impossible for its column -> treat it as missing, then fill 
# (DOING THIS FIRST SO THAT THE NEGATIVE INCOME GETS FILLED WITH THE MEDIAN)
df.loc[df['income'] <= 0, 'income'] = np.nan

# HINT 1: a numeric column has missing values -> decide how to fill them (mean? median? why?)
# Data prior to imputing
def missing_values():
    print("Missing values per column:")
    null_values = df.isna().sum()
    print(null_values.to_string())

    print("\nPercentage of missing values:")
    pct_missing = null_values.sum()/(len(df) * len(df.columns))
    print(f"{pct_missing * 100:.2F}%")

    print("\nNumber of rows WITHOUT any missing values:")
    rows_with_missing = len((df[(df.customer_id.isna()) | (df.income.isna())]))
    print(len(df) - rows_with_missing)

print("\n--------------BEFORE IMPUTING --------------")
missing_values()
df.head(10)

df['income'] = df['income'].fillna(df['income'].median())

# Data after imputing
print("\n--------------AFTER IMPUTING --------------")
missing_values()

# HINT 2: one category label is written in the wrong case -> standardize it

#Finding different formats for regions in the dataset
print("\nDistinct Region Values:", df["region"].nunique())
print(sorted(df["region"].dropna().unique(), reverse=True)[:20])

#Replacing east to East
df["region"] = df["region"].replace(["east"], "East")

#Checking To see If missing values are gone
print("\nDistinct Sex Values After Cleaning:", df["region"].nunique())
print(sorted(df["region"].dropna().unique(), reverse=True)[:20])

print(df.head(30))


--------------BEFORE IMPUTING --------------
Missing values per column:
customer_id    0
age            0
income         3
spending       0
region         0
subscribed     0

Percentage of missing values:
1.25%

Number of rows WITHOUT any missing values:
37

--------------AFTER IMPUTING --------------
Missing values per column:
customer_id    0
age            0
income         0
spending       0
region         0
subscribed     0

Percentage of missing values:
0.00%

Number of rows WITHOUT any missing values:
40

Distinct Region Values: 4
['east', 'West', 'South', 'East']

Distinct Sex Values After Cleaning: 3
['West', 'South', 'East']
    customer_id  age  income  spending region subscribed
0             1   25    61.3      39.1   West         No
1             2   25    54.9      42.1  South        Yes
2             3   51   105.1      59.9  South         No
3             4   39    84.7      49.9  South         No
4             5   43    72.4      48.3   East         No
5             6

**Explain each issue:** what you found and how you handled it.

*Your answer:* 
- The first issue I noticed was that tere were some missing income values. To fix this I found the column that had missing data. Then I filled the data with the median income in the data to get the best representation.

- The second issue was that some of the region data was not standaridzed. All of the regions started with the first letter capitalized, except for one data "east". So then I made sure the east it would be convereted to "East".

- The last issue was that some income values were negative, which is an issue beause income cant be negative. To fix this I made the value NA then filled in the median for it.


## Part 1: Pick the tool for each question

For each question: (a) name the data types involved, (b) name the tool, (c) run it, (d) answer in one plain sentence.

### Q1. Do customers who earn more also spend more?

In [3]:
# data types:  income: float, spending = float, customer string
# tool: regression in ols
#CHANGE TO REGRESSION IN OLS
# hint: two continuous variables moving together
#skip needs ols
X = df[["customer_id", "income"]]
X = sm.add_constant(X)

y = df["spending"]

model = sm.OLS(y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:               spending   R-squared:                       0.516
Model:                            OLS   Adj. R-squared:                  0.490
Method:                 Least Squares   F-statistic:                     19.76
Date:                Wed, 07 Oct 2026   Prob (F-statistic):           1.45e-06
Time:                        10:05:12   Log-Likelihood:                -131.25
No. Observations:                  40   AIC:                             268.5
Df Residuals:                      37   BIC:                             273.6
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const          18.4753      5.869      3.148      

a) data types: region = str, subscribed = string
b) name the tool
c) code:
d) answer: customers who earn more do spend more because the correlatoin is very high and the p-value is low meaning there is a low chance that this happened to just chance.

### Q2. Is a customer's region related to whether they subscribed?

In [4]:
# data types: region = str, ssubscribed = string
# tool: chi2_contingency to build a contingency table, and pandas crosstab to the place the two variables together.
# hint: two categorical variables -> build a table with pd.crosstab, then test it
import pandas as pd
from scipy.stats import chi2_contingency

#cross tab the varibles
table = pd.crosstab(df['region'], df['subscribed'])

# Perform the chi-square test
chi2, p, dof, expected = chi2_contingency(table, correction=False)

# Display the contingency table
print("Contingency Table:")
print(table)

chi_square_statistic = round(chi2, 2)
p_value = round(p,5)

# Display the test results
print(f"\nChi-square statistic: {chi_square_statistic}")
print(f"Degrees of freedom: {dof}")
print(f"P-value (statistical significance): {p_value}")

# Determine statistical significance
if p < 0.05:
    print("The test is statistically significant.")
else:
    print("The test is not statistically significant.")


Contingency Table:
subscribed  No  Yes
region             
East         5   16
South        6    3
West         9    1

Chi-square statistic: 13.16
Degrees of freedom: 2
P-value (statistical significance): 0.00139
The test is statistically significant.


*Your answer:*
Based on the Chi-square test results, we reject the null hypothesis and find a meaningful connection between a customer's location and their subscription choice

### Q3. Can we predict spending from income and age together? How much do they explain? (You can review your Assignment 4 for this)

In [5]:
# data types: predicting a continuous outcome from TWO predictors age = int, income = float
# tool: ols model
# hint: remember to add a constant; look at R-squared and each predictor's p-value
#skip needs pls
X = df[["income", "age"]]
X = sm.add_constant(X)

y = df["spending"]

model = sm.OLS(y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:               spending   R-squared:                       0.520
Model:                            OLS   Adj. R-squared:                  0.494
Method:                 Least Squares   F-statistic:                     20.04
Date:                Wed, 07 Oct 2026   Prob (F-statistic):           1.27e-06
Time:                        10:05:12   Log-Likelihood:                -131.11
No. Observations:                  40   AIC:                             268.2
Df Residuals:                      37   BIC:                             273.3
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         19.2622      6.000      3.210      0.0

*Your answer:*
Spending can indeed be predicted using a combination of income and age, as these variables account for 52.0% of the overall variation ($R^2 = 0.520$). Income proves to be a strong predictor ($p = 0.003$), but age is not statistically significant ($p = 0.515$).

### Q4. Which region has the most customers, and what is the average income overall?

In [6]:
# hint: count a categorical column
# take the mean of a numeric column
#a) data types involved: region = string, customer_id = int, income = float
#b) tool: Pandas .value_counts and .mean
print(df["region"].value_counts())
avg_income = round(df["income"].mean(), 2)
print(f"Average income: ${avg_income * 1000}")
#i think i did it right

region
East     21
West     10
South     9
Name: count, dtype: int64
Average income: $86460.0


*Your answer:*
The East region has the most customers, and their average income is 86460.

## Part 2: The reveal

Answer these two in a sentence each (no code needed).

**a)** Look back at your four tools (Correlation, Chi-square test, Multiple regression, & Descriptive statistic)). What decided which tool you used each time? (Hint: it was not the wording of the question.)

*Your answer:*
The specific data types of the variables dictated which analytical tool to apply

**b)** In Q1 you found a relationship. Can you tell the company that higher income *causes* higher spending? Why or why not?

*Your answer:*
Higher earnings do not necessarily cause higher spending, as correlation alone cannot prove a cause-and-effect relationship.

## Bonus

In Q2 you ran a chi-square test. Look at its **expected counts** (not the observed counts). Do they all meet the assumption you learned in the chi-square module? What would you tell the company about how much to trust that result?

In [7]:
# hint: chi2_contingency returns the expected table as its 4th value
# check: are all expected counts >= 5?
antincipated_df = pd.DataFrame(expected, index = table.index, columns = table.columns)
print(antincipated_df)

print("All expected counts >=5", (expected >=5).all())

subscribed    No   Yes
region                
East        10.5  10.5
South        4.5   4.5
West         5.0   5.0
All expected counts >=5 False


*Your answer:*
Since some expected counts fall below the threshold of five, the analysis is statistically unreliable given the small sample size, and the firm cannot rely on these conclusions.

### Push your code to GitHub